# Submission 1: Wine Quality Classification

**Nama:** Ridho Rezky Anwar  
**Username Dicoding:** ridhorezkyanwar  

## Deskripsi Dataset

Dataset yang digunakan adalah [Wine Quality Dataset](https://archive.ics.uci.edu/ml/datasets/wine+quality) dari UCI Machine Learning Repository.

- Jumlah data: 6497 baris, terdiri dari 1599 data red wine dan 4898 data white wine.
- Fitur: 11 fitur numerik, yaitu fixed acidity, volatile acidity, citric acid, residual sugar, chlorides, free sulfur dioxide, total sulfur dioxide, density, pH, sulphates, dan alcohol.
- Label: `quality` dengan skor 0 sampai 10.
- Label diubah menjadi dua kelas: `good` untuk quality >= 6 dan `bad` untuk quality < 6.

## Masalah

Penilaian kualitas wine secara manual membutuhkan waktu dan biaya. Proyek ini bertujuan memprediksi apakah wine termasuk kelas `good` atau `bad` berdasarkan komposisi kimianya.

## Solusi Machine Learning

Solusi yang dibuat adalah model binary classification berbasis TensorFlow yang dilatih dan dievaluasi melalui pipeline TensorFlow Extended (TFX) dengan Apache Beam sebagai orchestrator.

Pipeline mencakup ExampleGen, StatisticsGen, SchemaGen, ExampleValidator, Transform, Tuner, Trainer, Evaluator, dan Pusher.

## Metode Pengolahan

- Data red wine dan white wine digabungkan menjadi satu dataset.
- Label `quality` dibinarisasi menjadi `good` (1) dan `bad` (0).
- Sebelas fitur numerik dinormalisasi menggunakan z-score melalui `tft.scale_to_z_score`.
- Dataset dibagi menjadi data training dan evaluation menggunakan ExampleGen.
- Hyperparameter dicari dengan RandomSearch pada komponen Tuner.

## Arsitektur Model

Model yang digunakan memiliki arsitektur berikut:

```text
Input (11 fitur)
-> Dense(64, relu) -> Dropout(0.3)
-> Dense(32, relu) -> Dropout(0.3)
-> Dense(1, sigmoid)
```

## Metrik Evaluasi

Performa model dievaluasi menggunakan:

- Binary Accuracy.
- AUC (Area Under the ROC Curve).
- Model hanya dapat di-push apabila Binary Accuracy memenuhi threshold minimum 0.70 pada Evaluator TFX.

## Performa Model

Nilai Binary Accuracy dan AUC harus diisi berdasarkan output komponen Evaluator setelah pipeline selesai dijalankan. Jangan mengisi bagian ini dengan nilai perkiraan.

- Binary Accuracy: **[isi nilai aktual dari output Evaluator]**
- AUC: **[isi nilai aktual dari output Evaluator]**
- Status blessing: **[blessed / not blessed]**

## Opsi Deployment

Model disajikan sebagai Flask API menggunakan Docker dan di-deploy pada Railway cloud platform.

- Environment variable: `SERVING_MODEL_DIR=serving_model`
- Endpoint health check: `GET /health`
- Endpoint prediksi: `POST /predict`
- Endpoint monitoring: `GET /metrics`

## Web App

Tautan web app: [Wine Quality API](https://wine-quality-mlops-production.up.railway.app)

Contoh endpoint model serving:

```text
https://wine-quality-mlops-production.up.railway.app/health
https://wine-quality-mlops-production.up.railway.app/predict
```


## Monitoring

Monitoring dilakukan menggunakan Prometheus dan Grafana. Metric yang dikumpulkan dari endpoint `/metrics` adalah:

- `prediction_requests_total`: jumlah request prediksi.
- `prediction_request_latency_seconds`: latency request prediksi.
- `prediction_result_total`: jumlah prediksi berdasarkan label `good` atau `bad`.

Lampirkan screenshot dashboard Grafana dengan nama file persis:

`ridhorezkyanwar-monitoring.png`

Screenshot harus menampilkan dashboard monitoring yang sudah berisi data request, latency, dan hasil prediksi. Simpan kedua screenshot tersebut di folder yang sama dengan notebook sebelum membuat ZIP submission.

## Setup Environment

In [113]:
import os
import urllib.request
import pandas as pd
import numpy as np

print('Setup selesai')

Setup selesai


## Download & Persiapan Dataset

In [114]:
import os

os.makedirs('data', exist_ok=True)

# Hanya buat wine_quality.csv jika belum ada
if not os.path.exists('data/wine_quality.csv'):
    red_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
    white_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv'
    urllib.request.urlretrieve(red_url, 'data/winequality-red.csv')
    urllib.request.urlretrieve(white_url, 'data/winequality-white.csv')
    red = pd.read_csv('data/winequality-red.csv', sep=';')
    white = pd.read_csv('data/winequality-white.csv', sep=';')
    df = pd.concat([red, white], ignore_index=True)
    df.columns = [c.replace(' ', '_') for c in df.columns]
    df.to_csv('data/wine_quality.csv', index=False)
    # Hapus file asli agar TFX tidak baca file dengan header berbeda
    os.remove('data/winequality-red.csv')
    os.remove('data/winequality-white.csv')
    print('Dataset berhasil dibuat.')
else:
    df = pd.read_csv('data/wine_quality.csv')
    print('Dataset sudah ada, skip download.')

print(f'Total data: {len(df)} baris')
print(f'Kolom: {list(df.columns)}')
print(f'\nDistribusi quality:')
print(df['quality'].value_counts().sort_index())

Dataset sudah ada, skip download.
Total data: 6497 baris
Kolom: ['fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality']

Distribusi quality:
3      30
4     216
5    2138
6    2836
7    1079
8     193
9       5
Name: quality, dtype: int64


In [115]:
# Simpan sebagai CSV untuk TFX ExampleGen
df.to_csv('data/wine_quality.csv', index=False)
print('Dataset disimpan ke data/wine_quality.csv')
print(f'Shape: {df.shape}')
df.head()

Dataset disimpan ke data/wine_quality.csv
Shape: (6497, 12)


,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


## Inisialisasi TFX Pipeline

In [116]:
import tfx
import tensorflow as tf

print(f'TFX version: {tfx.__version__}')
print(f'TensorFlow version: {tf.__version__}')

TFX version: 1.12.0
TensorFlow version: 2.11.0


In [117]:
from tfx.components import (
    CsvExampleGen, StatisticsGen, SchemaGen,
    ExampleValidator, Transform, Trainer, Evaluator, Pusher
)
from tfx.components import Tuner
from tfx.dsl.components.common.resolver import Resolver
from tfx.dsl.experimental.latest_blessed_model_resolver import LatestBlessedModelResolver
from tfx.proto import example_gen_pb2, trainer_pb2, pusher_pb2
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing
from tfx.orchestration import metadata, pipeline
from tfx.orchestration.beam.beam_dag_runner import BeamDagRunner
import tensorflow_model_analysis as tfma

PIPELINE_NAME = 'ridhorezkyanwar-pipeline'
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, 'data')
PIPELINE_ROOT = os.path.join(BASE_DIR, PIPELINE_NAME)
METADATA_PATH = os.path.join(PIPELINE_ROOT, 'metadata', 'metadata.db')
SERVING_MODEL_DIR = os.path.join(BASE_DIR, 'serving_model')
TRANSFORM_MODULE = os.path.join(BASE_DIR, 'modules', 'transform.py')
TRAINER_MODULE = os.path.join(BASE_DIR, 'modules', 'trainer.py')
TUNER_MODULE = os.path.join(BASE_DIR, 'modules', 'tuner.py')

print('Konfigurasi pipeline:')
print(f'  Pipeline root: {PIPELINE_ROOT}')
print(f'  Data dir: {DATA_DIR}')
print(f'  Serving dir: {SERVING_MODEL_DIR}')

Konfigurasi pipeline:
  Pipeline root: d:\ProyekPengembangandanPengoperasianSistemMachineLearning\ridhorezkyanwar-pipeline
  Data dir: d:\ProyekPengembangandanPengoperasianSistemMachineLearning\data
  Serving dir: d:\ProyekPengembangandanPengoperasianSistemMachineLearning\serving_model


## Komponen 1: ExampleGen

In [118]:
output_config = example_gen_pb2.Output(
    split_config=example_gen_pb2.SplitConfig(
        splits=[
            example_gen_pb2.SplitConfig.Split(name='train', hash_buckets=8),
            example_gen_pb2.SplitConfig.Split(name='eval', hash_buckets=2),
        ]
    )
)
example_gen = CsvExampleGen(input_base=DATA_DIR, output_config=output_config)
print('ExampleGen: membagi data 80% train, 20% eval')

ExampleGen: membagi data 80% train, 20% eval


## Komponen 2: StatisticsGen

In [119]:
statistics_gen = StatisticsGen(examples=example_gen.outputs['examples'])
print('StatisticsGen: menghitung statistik deskriptif dataset')

StatisticsGen: menghitung statistik deskriptif dataset


## Komponen 3: SchemaGen

In [120]:
schema_gen = SchemaGen(
    statistics=statistics_gen.outputs['statistics'],
    infer_feature_shape=True
)
print('SchemaGen: membuat schema otomatis dari statistik dengan shape fitur tetap')

SchemaGen: membuat schema otomatis dari statistik dengan shape fitur tetap


## Komponen 4: ExampleValidator

In [121]:
example_validator = ExampleValidator(
    statistics=statistics_gen.outputs['statistics'],
    schema=schema_gen.outputs['schema'],
)
print('ExampleValidator: validasi data terhadap schema')

ExampleValidator: validasi data terhadap schema


## Komponen 5: Transform

In [122]:
transform = Transform(
    examples=example_gen.outputs['examples'],
    schema=schema_gen.outputs['schema'],
    module_file=TRANSFORM_MODULE,
)
print('Transform: z-score normalization + binarisasi label')

Transform: z-score normalization + binarisasi label


## Komponen 6: Tuner (Hyperparameter Tuning)

In [123]:
tuner = Tuner(
    module_file=TUNER_MODULE,
    examples=transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    train_args=trainer_pb2.TrainArgs(num_steps=100),
    eval_args=trainer_pb2.EvalArgs(num_steps=50),
)
print('Tuner: RandomSearch dengan 5 trials')

Tuner: RandomSearch dengan 5 trials


## Komponen 7: Trainer

In [124]:
trainer = Trainer(
    module_file=TRAINER_MODULE,
    examples=transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    hyperparameters=tuner.outputs['best_hyperparameters'],
    train_args=trainer_pb2.TrainArgs(num_steps=100),
    eval_args=trainer_pb2.EvalArgs(num_steps=50),
)
print('Trainer: melatih model dengan hyperparameter terbaik dari Tuner')

Trainer: melatih model dengan hyperparameter terbaik dari Tuner


## Komponen 8: Resolver

In [125]:
model_resolver = Resolver(
    strategy_class=LatestBlessedModelResolver,
    model=Channel(type=Model),
    model_blessing=Channel(type=ModelBlessing),
).with_id('latest_blessed_model_resolver')
print('Resolver: mendapatkan model terbaik yang sudah di-bless sebelumnya')

Resolver: mendapatkan model terbaik yang sudah di-bless sebelumnya


## Komponen 9: Evaluator

In [126]:
eval_config = tfma.EvalConfig(
    model_specs=[
        tfma.ModelSpec(
            signature_name='serving_default',
            label_key='quality_xf',
            preprocessing_function_names=['transform_features'],
        )
    ],
    slicing_specs=[tfma.SlicingSpec()],
    metrics_specs=[
        tfma.MetricsSpec(
            metrics=[
                tfma.MetricConfig(class_name='BinaryAccuracy'),
                tfma.MetricConfig(class_name='AUC'),
                tfma.MetricConfig(
                    class_name='BinaryAccuracy',
                    threshold=tfma.MetricThreshold(
                        value_threshold=tfma.GenericValueThreshold(
                            lower_bound={'value': 0.7}
                        ),
                        change_threshold=tfma.GenericChangeThreshold(
                            direction=tfma.MetricDirection.HIGHER_IS_BETTER,
                            absolute={'value': -0.01},
                        ),
                    ),
                ),
            ]
        )
    ],
)

evaluator = Evaluator(
    examples=example_gen.outputs['examples'],
    model=trainer.outputs['model'],
    baseline_model=model_resolver.outputs['model'],
    eval_config=eval_config,
)
print('Evaluator: threshold BinaryAccuracy >= 0.70')

Evaluator: threshold BinaryAccuracy >= 0.70


## Komponen 10: Pusher

In [127]:
pusher = Pusher(
    model=trainer.outputs['model'],
    model_blessing=evaluator.outputs['blessing'],
    push_destination=pusher_pb2.PushDestination(
        filesystem=pusher_pb2.PushDestination.Filesystem(
            base_directory=SERVING_MODEL_DIR
        )
    ),
)
print('Pusher: push model ke serving_model directory')

Pusher: push model ke serving_model directory


## Jalankan Pipeline

In [128]:
components = [
    example_gen,
    statistics_gen,
    schema_gen,
    example_validator,
    transform,
    tuner,
    trainer,
    model_resolver,
    evaluator,
    pusher,
]

tfx_pipeline = pipeline.Pipeline(
    pipeline_name=PIPELINE_NAME,
    pipeline_root=PIPELINE_ROOT,
    components=components,
    metadata_connection_config=metadata.sqlite_metadata_connection_config(METADATA_PATH),
    enable_cache=True,
)

BeamDagRunner().run(tfx_pipeline)

Trial 5 Complete [00h 00m 01s]
val_binary_accuracy: 0.7138485312461853

Best val_binary_accuracy So Far: 0.7291507124900818
Total elapsed time: 00h 00m 06s
Results summary
Results in d:\ProyekPengembangandanPengoperasianSistemMachineLearning\ridhorezkyanwar-pipeline\Tuner\.system\executor_execution\50\.temp\50\wine_quality_tuning
Showing 10 best trials
Objective(name="val_binary_accuracy", direction="max")

Trial 2 summary
Hyperparameters:
units: 32
dropout_rate: 0.2
learning_rate: 0.01
Score: 0.7291507124900818

Trial 1 summary
Hyperparameters:
units: 96
dropout_rate: 0.30000000000000004
learning_rate: 0.01
Score: 0.7276204824447632

Trial 4 summary
Hyperparameters:
units: 64
dropout_rate: 0.5
learning_rate: 0.001
Score: 0.7138485312461853

Trial 0 summary
Hyperparameters:
units: 128
dropout_rate: 0.5
learning_rate: 0.001
Score: 0.7061973810195923

Trial 3 summary
Hyperparameters:
units: 64
dropout_rate: 0.4
learning_rate: 0.0001
Score: 0.5371078848838806


100/100 [==============================] - 1s 3ms/step - loss: 0.5776 - binary_accuracy: 0.7016 - val_loss: 0.5309 - val_binary_accuracy: 0.7200
INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


...

INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:Assets written to: d:\ProyekPengembangandanPengoperasianSistemMachineLearning\ridhorezkyanwar-pipeline\Trainer\model\51\Format-Serving\assets


INFO:tensorflow:Assets written to: d:\ProyekPengembangandanPengoperasianSistemMachineLearning\ridhorezkyanwar-pipeline\Trainer\model\51\Format-Serving\assets


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


## 5. Performa Model

Setelah pipeline selesai dijalankan, model dievaluasi dengan metrik:
- **Binary Accuracy:** dilihat dari output Evaluator
- **AUC:** dilihat dari output Evaluator
- Model di-push ke `serving_model/` jika Binary Accuracy â‰¥ 0.70

## 6. Deployment

Model di-deploy menggunakan **Flask API** yang berjalan di **Railway** cloud platform.
- Endpoint prediksi: `POST /predict`
- Endpoint metrics: `GET /metrics`
- Endpoint health: `GET /health`

## 7. Monitoring

Sistem dimonitor menggunakan **Prometheus** yang mengumpulkan metrics dari endpoint `/metrics`:
- `prediction_requests_total` - total request
- `prediction_request_latency_seconds` - latency
- `prediction_result_total` - distribusi hasil prediksi

Dashboard monitoring divisualisasikan menggunakan **Grafana**.

In [129]:
# Cek hasil serving model
import os
if os.path.exists('serving_model'):
    versions = os.listdir('serving_model')
    print(f'Model berhasil di-push. Versi tersedia: {versions}')
else:
    print('serving_model directory belum ada - pipeline belum selesai atau model tidak lolos threshold')

Model berhasil di-push. Versi tersedia: ['1788755731']
